<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi-e2/notebooks/gsm8k/E2_Experiment_SRIG_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Experiment 2: Two-Step Lookahead MEDAL vs. AB-MEDAL on GSM8K with Sum of Relative Information Gains (SRIG) — Test Split (200 Primary + 300 Extended Questions)

This notebook runs **Experiment 2 (E2)** on the predetermined GSM8K `test` split with `LLaDA-8B-Instruct` using the **Two-Step SRIG / $\text{Future}R_{ig}$** reward function:
For each candidate child state $s_i$, search projects the highest-confidence action $a^*_i \in A(s_i)$ to form virtual grandchild $g_i^* = s_i \oplus a^*_i$, computing the two-step sum of relative information gains:
$$
\text{Future}R_{ig}(s, a_i) = R_{ig}(s, s_i) + \lambda \, R_{ig}(s_i, g_i^*) = \frac{H(s) - H(s_i)}{\max(H(s), \epsilon)} + \lambda \frac{H(s_i) - H(g_i^*)}{\max(H(s_i), \epsilon)}
$$

### Test Dataset Split Organization (Aligned with E0 & E1 Test Runs):
1. **Primary 200 Test Samples** (`sample_slice=(0, 200)`): The exact 200 predetermined test questions from `configs/gsm8k/experiment_samples.json` (`sample_seed=1`).
2. **Extended 300 Test Samples** (`sample_slice=(200, 500)`): 300 additional held-out test questions from `configs/gsm8k/experiment_samples_500.json` under the identical `sample_seed=1` ordering.
3. **Combined 500 Test Samples**: Full 500-question test pool evaluation, Wilson 95% confidence intervals, and paired McNemar statistical tests.

### Guardrails & Protocols:
- **Matched Search Budgets**: Continuous search to $B = 512$ with prefix checkpoints at $B \in \{256, 512\}$ across both **MEDAL** and **AB-MEDAL**.
- **Simulation Cap Safeguard ($4B$)**: Capped at $4B$ simulations per question as an exploration runaway safeguard; actual NFE utilization is logged.
- **Virtual Grandchildren ($g_i^*$)**: Evaluated and cached in `cached_analyses` but never attached to the tree or assigned GEN/CONT posteriors.
- **Terminal vs. Actionless Children**: Terminal children ($0$ masks remaining) have $H(s_i) = 0$ and $R_{ig}(s_i, g_i^*) = 0$. Actionless non-terminal children fallback to 1-step $R_{ig}(s, s_i)$ without fabricating $H(g) = 0$.
- **Exact Checkpoint Boundaries**: Non-mutating hypothetical boundary snapshots ensure standalone $B=256$ equivalence without altering live tree execution toward $B=512$.
- **Cooperative Varlen FlashAttention-2 Batching & Incremental Google Drive Resumption**: Packs active states across questions in each batch (`BATCH_SIZE = 2` on L4, `4` or `6` on A100) and saves per-question artifacts after every batch so interrupted sessions resume automatically with 0 redundant forward passes.


## 1. Verify GPU Hardware
Verify CUDA GPU supporting bfloat16 (recommended: L4 or A100 with $\ge 22$ GiB VRAM).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA GPU runtime in Colab (L4 or A100), then reconnect.')
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024 ** 3)
print(f'GPU: {gpu.name} | Total VRAM: {vram_gb:.2f} GiB')
if vram_gb < 20.0:
    raise RuntimeError('This notebook uses unquantized BF16 weights. Select an L4 or A100 GPU runtime.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('This GPU does not support bfloat16 inference.')
print('✓ GPU hardware verification passed.')


## 2. Repository Setup
Clone or update the GitHub repository on branch `adi-e2`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/CooL-Legend/CSCI544-DLM.git'
BRANCH = 'adi-e2'
REPO_DIR = Path('/content/CSCI544-DLM') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        token = os.environ.get('GITHUB_TOKEN', '')
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get('GITHUB_TOKEN')
            except Exception:
                pass
        clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
        subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR / 'src') not in sys.path:
        sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Working directory: {Path.cwd()}')


## 3. Install Dependencies
Install pinned dependencies and editable package.

In [ ]:
import site
import subprocess
import sys

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements/llada-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
    import importlib
    importlib.invalidate_caches()
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
print('Dependencies installed successfully.')


## 4. Run Algorithmic Pre-Flight Unit Tests
Verify 2-step lookahead SRIG reward computation, test-split alignment (200 primary and 500 extended manifests), state caching, and batched parity before loading weights.

In [ ]:
import subprocess
import sys

res = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/tasks/test_experiment_2.py', '-v'],
    capture_output=True, text=True
)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError('Pre-flight E2 unit tests failed!')
print('All E2 algorithmic unit tests passed.')


## 5. Mount Google Drive & Configure Cache
Mount Google Drive for persistent artifact and model caching.

In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path('/content/drive/MyDrive/CSCI544-DLM')
OUTPUT_DIR = Path('results/e2_srig_test')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = DRIVE_ROOT / 'results/e2_srig_test'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cache_dir = DRIVE_ROOT / 'cache/huggingface'
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

print(f'Output directory: {OUTPUT_DIR.resolve()}')


## 6. Experiment Controls & Staged Model Loading
Configure `test` stage (200 primary + 300 extended = 500 test questions), budgets [256, 512], depth 10, batch size 2, SRIG reward mode ($\lambda=1.0$), and load `LLaDA-8B-Instruct` from staged NVMe storage.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.experiment_2 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context, load_model
)

STAGE = 'test'
METHODS = ('medal', 'ab_medal')
BUDGETS = (256, 512)
REWARD_MODE = 'srig'
REWARD_LAMBDA = 1.0
SIM_CAP_MULTIPLIER = 4
# On NVIDIA L4 (22.4 GiB VRAM), BATCH_SIZE = 2 is recommended for stability;
# BATCH_SIZE = 4 or 6 is suited for A100 (40/80 GB) or High-RAM runtimes.
BATCH_SIZE = 2
RESUME = True

# Load base context on the 500-question test split to initialize model identity
base_ctx = context(
    stage=STAGE,
    methods=METHODS,
    budgets=BUDGETS,
    sample_slice=(0, 500),
    reward_mode=REWARD_MODE,
    reward_lambda=REWARD_LAMBDA,
    sim_cap_multiplier=SIM_CAP_MULTIPLIER,
)

profile = get_profile('llada')
model_name = profile['repo_id'].split('/')[-1]
drive_model_dir = DRIVE_ROOT / 'models' / model_name
runtime_model_dir = Path('/content/models') / model_name

# Fast local NVMe cache in Colab runtime if available on Drive (or cache to Drive first)
if IN_COLAB:
    if not (drive_model_dir / 'config.json').is_file() and not (runtime_model_dir / 'config.json').is_file():
        subprocess.run([
            sys.executable, '-u', str(REPO_DIR / 'scripts/cache_model.py'),
            '--model', 'llada', '--drive-dir', str(drive_model_dir),
            '--runtime-dir', str(runtime_model_dir),
        ], check=True)
    elif drive_model_dir.is_dir() and not (runtime_model_dir / 'config.json').is_file():
        print(f'Copying model from Google Drive to local fast runtime storage: {runtime_model_dir}...')
        shutil.copytree(drive_model_dir, runtime_model_dir)

MODEL_PATH = str(runtime_model_dir if runtime_model_dir.is_dir() else (drive_model_dir if drive_model_dir.is_dir() else profile['repo_id']))
print('Using model location:', MODEL_PATH)
print('Loading LLaDA-8B-Instruct wrapper on GPU in bfloat16...')
wrapper = load_model(base_ctx, model_path=MODEL_PATH, device='cuda', local_files_only=Path(MODEL_PATH).is_dir())
print(f'✓ Model loaded: {wrapper.identity["family"]} ({wrapper.identity["repo_id"]} @ {wrapper.identity["revision"]})')
print(f'✓ Stage: {STAGE} | Methods: {METHODS} | Budgets: {BUDGETS} | Reward: {REWARD_MODE} (λ={REWARD_LAMBDA}) | Sim Cap: {SIM_CAP_MULTIPLIER}B | Total Test Pool: {len(base_ctx["examples"])} questions | Batch Size: {BATCH_SIZE}')


## 7. Execute Part 1: Primary Test Set (Original 200 Test Questions)
Evaluates the exact 200 predetermined test questions (`sample_slice=(0, 200)`) from `configs/gsm8k/experiment_samples.json`.

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_2 import run_e2_stage

ctx_200 = context(
    stage=STAGE,
    methods=METHODS,
    budgets=BUDGETS,
    sample_slice=(0, 200),
    reward_mode=REWARD_MODE,
    reward_lambda=REWARD_LAMBDA,
    sim_cap_multiplier=SIM_CAP_MULTIPLIER,
)
ctx_200['setup'] = dict(base_ctx.get('setup', {}))
print('Starting Part 1 (Primary 200 Test Questions) with E2 SRIG reward...')
t0 = perf_counter()
q_rows_200, agg_rows_200 = run_e2_stage(
    wrapper,
    ctx_200,
    OUTPUT_DIR / 'primary_200',
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
print(f'Part 1 Complete in {(perf_counter() - t0) / 60:.2f} minutes ({len(q_rows_200)} checkpoint outcomes).')


## 8. Execute Part 2: Extended Test Set (Additional 300 Test Questions)
Evaluates the additional 300 predetermined test questions (`sample_slice=(200, 500)`) from `configs/gsm8k/experiment_samples_500.json`.

In [ ]:
ctx_300 = context(
    stage=STAGE,
    methods=METHODS,
    budgets=BUDGETS,
    sample_slice=(200, 500),
    reward_mode=REWARD_MODE,
    reward_lambda=REWARD_LAMBDA,
    sim_cap_multiplier=SIM_CAP_MULTIPLIER,
)
ctx_300['setup'] = dict(base_ctx.get('setup', {}))
print('Starting Part 2 (Extended 300 Test Questions) with E2 SRIG reward...')
t0 = perf_counter()
q_rows_300, agg_rows_300 = run_e2_stage(
    wrapper,
    ctx_300,
    OUTPUT_DIR / 'extended_300',
    methods=METHODS,
    budgets=BUDGETS,
    max_depth=MAX_SEARCH_DEPTH,
    num_candidates=MIN_CANDIDATES,
    resume=RESUME,
    batch_size=BATCH_SIZE,
)
print(f'Part 2 Complete in {(perf_counter() - t0) / 60:.2f} minutes ({len(q_rows_300)} checkpoint outcomes).')


## 9. Results & Paired Comparisons (Primary 200, Extended 300, and Combined 500 Test Questions)
Display aggregate accuracy, Wilson 95% confidence intervals, NFE utilization, and paired McNemar exact tests comparing AB-MEDAL vs. MEDAL across Primary 200, Extended 300, and Combined 500 test questions.

In [ ]:
import pandas as pd
from IPython.display import display
from scripts.run_e2 import compute_paired_comparison
from dlm_project.experiments.sweep_report import write_csv
from dlm_project.tasks.gsm8k.experiment_2 import compute_e2_aggregate_rows

summaries_dir = OUTPUT_DIR / 'summaries'
summaries_dir.mkdir(parents=True, exist_ok=True)

agg_cols = [
    'search_budget', 'method', 'sample_count', 'accuracy_percent',
    'ci_95_low', 'ci_95_high', 'parse_rate', 'mean_nfe_utilization',
    'mean_search_nfe', 'mean_completion_nfe', 'mean_total_nfe',
    'mean_selected_depth', 'mean_total_wall_seconds',
]
paired_cols = [
    'search_budget', 'paired_questions', 'medal_accuracy', 'ab_medal_accuracy',
    'accuracy_delta_percent', 'ab_medal_only_correct', 'medal_only_correct', 'mcnemar_exact_p',
]

combined_q_rows = q_rows_200 + q_rows_300
combined_agg_rows = compute_e2_aggregate_rows(combined_q_rows, STAGE, METHODS, BUDGETS)

for title, rows, agg, slug in [
    ('Part 1: Primary 200 Test Questions', q_rows_200, agg_rows_200, 'primary_200'),
    ('Part 2: Extended 300 Test Questions', q_rows_300, agg_rows_300, 'extended_300'),
    ('Combined Pool: All 500 Test Questions', combined_q_rows, combined_agg_rows, 'combined_500'),
]:
    if rows:
        write_csv(summaries_dir / f'e2_srig_{slug}_per_question.csv', rows)
    if agg:
        write_csv(summaries_dir / f'e2_srig_{slug}_aggregate.csv', agg)
        df_agg = pd.DataFrame(agg)
        print(f'=== {title} Aggregate Metrics ===')
        display(df_agg[[c for c in agg_cols if c in df_agg.columns]])
    paired = compute_paired_comparison(rows, budgets=BUDGETS)
    if paired:
        df_paired = pd.DataFrame(paired)
        print(f'=== {title} Paired Comparison (AB-MEDAL vs. MEDAL) ===')
        display(df_paired[[c for c in paired_cols if c in df_paired.columns]])
        write_csv(summaries_dir / f'e2_srig_{slug}_paired_comparison.csv', paired)

if combined_q_rows:
    write_csv(summaries_dir / 'e2_srig_test_per_question.csv', combined_q_rows)
    write_csv(summaries_dir / 'e2_test_per_question.csv', combined_q_rows)
if combined_agg_rows:
    write_csv(summaries_dir / 'e2_srig_test_aggregate.csv', combined_agg_rows)
    write_csv(summaries_dir / 'e2_test_aggregate.csv', combined_agg_rows)
paired_combined = compute_paired_comparison(combined_q_rows, budgets=BUDGETS)
if paired_combined:
    write_csv(summaries_dir / 'e2_srig_test_paired_comparison.csv', paired_combined)
    write_csv(summaries_dir / 'e2_test_paired_comparison.csv', paired_combined)


## 10. Interactive Search Tree & Decision Trace Inspector
Inspect the search tree topology, candidate nodes, and decision trace for any evaluated test question.

In [ ]:
from dlm_project.tasks.gsm8k.tree_viz import print_ascii_tree, summarize_decision_trace
import json

INSPECT_SPLIT = 'primary_200'  # 'primary_200' or 'extended_300'
INSPECT_QUESTION_IDX = 0
INSPECT_BUDGET = 256  # 256 or 512

split_ctx = ctx_200 if INSPECT_SPLIT == 'primary_200' else ctx_300
sample_ex_id = split_ctx['examples'][INSPECT_QUESTION_IDX]['example_id']
chk_dir = OUTPUT_DIR / INSPECT_SPLIT / STAGE / 'ab_medal' / sample_ex_id / f'checkpoints/{INSPECT_BUDGET}'
tree_json = chk_dir / 'tree.json'
metrics_json = chk_dir / 'metrics.json'

if tree_json.is_file():
    print(f'=== AB-MEDAL Search Tree at B={INSPECT_BUDGET} for {sample_ex_id} ===')
    print(print_ascii_tree(tree_json, max_depth=5, max_children_per_node=3))

if metrics_json.is_file():
    met = json.loads(metrics_json.read_text())
    print(f'=== Checkpoint Metrics ({sample_ex_id} @ B={INSPECT_BUDGET}) ===')
    print(f'  Selected Node: {met["selected_node_id"]} | Depth: {met["selected_depth"]} | Q: {met["selected_q"]:.4f}')
    print(f'  Exact Match: {met["exact_match"]} | Pred: {met["predicted_answer"]} | Ref: {met["reference_answer"]}')
    print(f'  NFE Utilization: {met.get("nfe_utilization", 1.0)*100:.1f}% | Sims: {met.get("simulations_completed", "N/A")}/{met.get("simulation_cap", "N/A")}')


## 11. E3 Offline Metric Extraction & Publication Figures
Run offline tree allocation, branching dynamics, and structural analysis on saved test checkpoints (zero GPU compute), and generate 6 publication figures.

In [ ]:
import shutil
from dlm_project.tasks.gsm8k.e3_analysis import analyze_e3_stage
from IPython.display import Image, display, Markdown

# Build combined trace directory linking primary_200 and extended_300
combined_results_dir = OUTPUT_DIR / 'combined_500' / STAGE
combined_results_dir.mkdir(parents=True, exist_ok=True)
for part_dir in [OUTPUT_DIR / 'primary_200' / STAGE, OUTPUT_DIR / 'extended_300' / STAGE]:
    if part_dir.is_dir():
        for m in METHODS:
            m_src = part_dir / m
            if m_src.is_dir():
                m_dst = combined_results_dir / m
                m_dst.mkdir(parents=True, exist_ok=True)
                for q_dir in m_src.iterdir():
                    if q_dir.is_dir() and not (m_dst / q_dir.name).exists():
                        try:
                            (m_dst / q_dir.name).symlink_to(q_dir.resolve(), target_is_directory=True)
                        except OSError:
                            shutil.copytree(q_dir, m_dst / q_dir.name)

# Select results directory: combined_500 if populated, else primary_200
has_combined = any((combined_results_dir / m).is_dir() and any((combined_results_dir / m).iterdir()) for m in METHODS)
test_results_dir = combined_results_dir if has_combined else (OUTPUT_DIR / 'primary_200' / STAGE)
e3_output_dir = OUTPUT_DIR / ('combined_500' if has_combined else 'primary_200') / 'summaries'
e3_output_dir.mkdir(parents=True, exist_ok=True)

print(f'Running E3 Offline Tree Analysis on {test_results_dir}...')
e3_q, e3_agg = analyze_e3_stage(
    results_dir=test_results_dir,
    output_dir=e3_output_dir,
    stage=STAGE,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
)
print(f'✓ E3 Analysis Completed successfully: {len(e3_q)} question outcomes evaluated.')

figures_dir = e3_output_dir / 'e3_figures'
fig_titles = {
    'fig1_nfe_by_depth': 'Figure 1: NFE Allocation by Tree Depth — MEDAL vs AB-MEDAL across budgets',
    'fig2_branching_factors': 'Figure 2: Branching Factor Distribution — Children per expanded parent',
    'fig3_candidate_density': 'Figure 3: Candidate Density vs Depth — Minimum-C availability and selection depth',
    'fig4_root_subtrees': 'Figure 4: Root-Subtree Compute Concentration — Search effort across root branches',
    'fig5_reward_redundancy': 'Figure 5: Reward and Search Redundancy — Negative IG rates and duplicate states',
    'fig6_accuracy_vs_budget': 'Figure 6: Accuracy vs Search Budget — Performance scaling curve',
}

for fig_key, title in fig_titles.items():
    png_path = figures_dir / f'{fig_key}.png'
    if png_path.is_file():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(png_path), width=750))


## 12. Synchronize All Artifacts to Google Drive
Flush and sync all in-flight buffers to Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Google Drive flushed successfully.')
